# 模型初始化参数

在 LangChain 中，无论是直接使用模型类（如 `ChatDeepSeek`、`ChatOpenAI`），还是使用统一接口 `init_chat_model`，都可以在初始化时传入参数来控制模型。API 文档：

https://docs.langchain.org.cn/oss/python/langchain/models#parameters

**模型参数远不止 `temperature` 和 `max_tokens` 两个**，按作用可以分成 4 类：

**① 连接类：调用哪个模型、怎么连上它**

| 参数 | 作用 | 不传时 |
|---|---|---|
| `model` | 模型名称，如 `deepseek-v4-flash`、`qwen-plus` | 必填 |
| `model_provider` | 模型提供商，如 `openai`、`deepseek`。设为 `"deepseek"` 时会自动加载 `langchain-deepseek`，底层调用 `ChatDeepSeek` 类 | 可写在 model 前缀里（如 `"deepseek:deepseek-v4-flash"`），或由 LangChain 根据模型名自动推断 |
| `api_key` | API 密钥，用于认证 | 从环境变量读取（如 `DEEPSEEK_API_KEY`） |
| `base_url` | API 请求地址 | 厂商官方地址 |

**② 生成控制类：模型"说什么、说多长"**（本笔记重点）

| 参数 | 作用 | 不传时（以 DeepSeek 为例） |
|---|---|---|
| `temperature` | 温度，控制输出的随机性 → 第 1 节 | 1.0 |
| `top_p` | 核采样，只在概率最高的一批词里挑 → 第 2 节 | 1.0 |
| `max_tokens` | 最多生成多少个 token → 第 3 节 | 8K（非思考模式）/ 64K（思考模式） |
| `stop` | 停止序列，生成到指定字符串就停 → 第 4 节 | 无 |
| `presence_penalty` / `frequency_penalty` | 重复惩罚，减少重复内容 → 第 5 节 | 0（DeepSeek V4 已废弃） |

**③ 网络可靠性类：请求失败怎么办** → 第 6 节

| 参数 | 作用 | 不传时 |
|---|---|---|
| `timeout` | 超时时间（秒） | 600 秒 |
| `max_retries` | 请求失败后的最大重试次数 | 2 次 |
| `rate_limiter` | 限流器，控制每秒请求数 | 不限流 |

**④ 厂商特有参数：某家模型独有的功能** → 第 7 节

如 DeepSeek 的思考模式开关，通过 `extra_body`、`reasoning_effort` 等方式传入。

> ⚠️ **关于默认值**：LangChain 中这些参数的默认值基本都是 `None`，意思是**"不把这个参数发给服务器"**，由服务器使用它自己的默认值（上表"不传时"一列就是实际效果）。
> - 网上常见的"temperature 默认 0.7"是早期版本 `ChatOpenAI` 的默认值，现在已改为 `None`；
> - LangChain 文档中写的"max_retries 默认 6"是通用说明。`ChatDeepSeek` / `ChatOpenAI` 底层使用 openai SDK，实际默认超时 600 秒、重试 2 次。
>
> 下面用代码验证一下：

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

# 除了连接信息，什么参数都不传，看看默认值
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
)
print("temperature:", model.temperature)
print("max_tokens:", model.max_tokens)
print("实际发给服务器的参数:", model._default_params)  # 内部属性，仅用于观察：值为 None 的参数不会被发送
print("超时:", model.root_client.timeout)  # root_client 是底层的 openai 客户端
print("重试次数:", model.root_client.max_retries)

temperature: None
max_tokens: None
实际发给服务器的参数: {'model': 'deepseek-v4-flash', 'stream': False}
超时: Timeout(connect=5.0, read=600, write=600, pool=600)
重试次数: 2


# 1. temperature（温度）

## 1.1 先弄懂：大模型是怎么"写字"的？

大模型生成文本时，是**一个 token 一个 token 往外蹦**的，每蹦一个都要经过三步：

1. **打分**：根据前面已有的内容，给词表里的每个候选 token 打一个分数；
2. **转成概率**：把分数换算成概率，所有候选的概率加起来等于 100%；
3. **抽签**：按概率随机抽一个 token 输出，拼到末尾，再预测下一个。

比如写到"春风拂过\_\_"时，候选词的概率可能是：

| 候选 | 柳 | 面 | 江 | 山 | 城 |
|:---:|:---:|:---:|:---:|:---:|:---:|
| 概率 | 50% | 25% | 15% | 7% | 3% |

"柳"的概率最高，但因为是**按概率抽签**，"面""江"也有机会被抽中。这就是**同一个问题问两遍，回答却不一样**的根本原因。

## 1.2 温度的作用：抽签之前，把概率分布"变陡"或"变平"

温度 $T$ 作用在第 2 步：先把每个候选的分数 $z_i$ 除以 $T$，再换算成概率：

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

公式不用记，记住效果即可：

| 温度 | 对概率分布的影响 | 输出特点 |
|---|---|---|
| $T = 0$ | 相当于每次都直接选概率最高的词 | 输出几乎固定 |
| $0 < T < 1$ | 分布变**陡**：强的更强，弱的更弱 | 保守、稳定、准确 |
| $T = 1$ | 保持模型原始的概率分布 | 正常水平 |
| $T > 1$ | 分布变**平**：冷门词被抽中的机会变大 | 多样、有创意；太高则容易胡言乱语 |

> 💡 **一句话理解：温度就是模型的"冒险程度"。** 温度越低越保守（总挑最稳的词），越高越敢冒险（冷门词也敢用）。
>
> "温度"这个名字来自物理学：温度越高，分子运动越混乱、越随机。
>
> 注意：温度**不会让模型变聪明或变笨**，它只改变"从候选词里怎么挑"。

## 1.3 模拟演示：温度如何改变概率分布

下面用纯 Python 模拟"春风拂过\_\_"这个例子（不调用 API，不花钱），看看不同温度下各候选词的概率，以及"抽签" 20 次的结果：

In [2]:
import random

# 写到"春风拂过__"时，模型给出的候选词及原始概率（示意数据）
candidates = {"柳": 0.50, "面": 0.25, "江": 0.15, "山": 0.07, "城": 0.03}

def apply_temperature(probs: dict, t: float) -> dict:
    """按温度 t 重新调整概率分布，等价于 softmax(分数 / t)"""
    scaled = {word: p ** (1 / t) for word, p in probs.items()}
    total = sum(scaled.values())
    return {word: v / total for word, v in scaled.items()}

random.seed(0)  # 固定随机种子，保证每次运行结果相同
for t in [0.1, 0.5, 1.0, 1.5, 2.0]:
    probs = apply_temperature(candidates, t)
    print(f"temperature = {t}")
    for word, p in probs.items():
        print(f"  {word} {'█' * round(p * 40):<40} {p:6.1%}")
    # 按调整后的概率"抽签" 20 次，模拟模型生成 20 次
    picks = random.choices(list(probs), weights=list(probs.values()), k=20)
    print(f"  抽签 20 次：{''.join(picks)}\n")

temperature = 0.1
  柳 ████████████████████████████████████████  99.9%
  面                                            0.1%
  江                                            0.0%
  山                                            0.0%
  城                                            0.0%
  抽签 20 次：柳柳柳柳柳柳柳柳柳柳柳柳柳柳柳柳柳柳柳柳

temperature = 0.5
  柳 █████████████████████████████             73.4%
  面 ███████                                   18.3%
  江 ███                                        6.6%
  山 █                                          1.4%
  城                                            0.3%
  抽签 20 次：柳柳面柳柳柳柳柳面江柳面柳面柳柳柳柳面柳

temperature = 1.0
  柳 ████████████████████                      50.0%
  面 ██████████                                25.0%
  江 ██████                                    15.0%
  山 ███                                        7.0%
  城 █                                          3.0%
  抽签 20 次：柳柳江柳柳江柳面柳山江柳柳柳面山柳面面面

temperature = 1.5
  柳 ████████████████                          40.0%


观察结果：

- **temperature=0.1**：分布极陡，"柳"占 99.9%，抽 20 次全是"柳" → 输出稳定、可预测；
- **temperature=1.0**：就是原始概率，以"柳"为主，其他词偶尔出现；
- **temperature=2.0**：分布被"拍平"，"山""城"这类冷门词频繁出现 → 输出多样，但也更容易离谱。

## 1.4 实验：调用 DeepSeek，对比 temperature=0 和 temperature=1.5

用同一个提示词，分别以 `temperature=0` 和 `temperature=1.5` 各调用 3 次：

In [5]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=0,
)
for i in range(3):
    print(model.invoke("帮我写一首描述春天的七言绝句诗").content)


《春日即景》  
莺梭燕剪弄春柔，柳眼初开雨乍收。  
一树梨花浑似雪，东风吹上木兰舟。
《春日即景》

昨夜东风过小楼，杏花春雨满汀洲。

啼莺唤醒垂杨梦，又报春光到陌头。
《春日即景》
春回大地绿初匀，万紫千红处处新。
蝶舞蜂飞迷曲径，风梳烟柳醉游人。


In [6]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-pro",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=1.5,
)
for i in range(3):
    print(model.invoke("帮我写一首描述春天的七言绝句诗").content)


《春日即景》

东风拂柳绿丝长，桃李争妍满院香。  
燕子归来寻旧垒，一溪春水映斜阳。
《春行》  
东风染柳绿丝长，  
桃李花开满院香。  
燕子归来寻旧垒，  
一湾碧水映斜阳。
《春日》
春入江南草木知，东风先上小桃枝。  
呢喃燕子归来早，又啄芹泥补旧篱。


## 1.5 🤔 奇怪：temperature=0 时，三次结果为什么也不一样？

按 1.2 的理论，`temperature=0` 时每一步都选概率最高的词，三次输出应该一模一样。可上面的实验中：

- `temperature=0` 的三首诗**各不相同**；
- `temperature=1.5` 的三首诗和 `temperature=0` 相比，看不出明显区别。

**原因：DeepSeek V4 默认开启了"思考模式"（thinking mode），而思考模式下 temperature 不生效。**

DeepSeek 官方 API 文档对 temperature 的说明是 "Has no effect in thinking mode"（思考模式下无效）。更坑的是，**传了也不会报错**，只会被悄悄忽略。所以上面两组实验的采样设置实际上完全一样，温度根本没起作用。

**怎么判断模型是否开启了思考模式？** 看返回结果中有没有 `reasoning_content`（思考过程）。第 3 节的输出中就能看到：

```
additional_kwargs={'reasoning_content': '我们需要回答中文。用户问“帮我讲解一下……'}
'completion_tokens_details': {..., 'reasoning_tokens': 40, ...}
```

> 📌 这不是 DeepSeek 独有的现象，**推理（思考）模型普遍会限制采样参数**。例如 OpenAI 的 GPT-5 推理模型只支持 `temperature=1`，LangChain 的 `ChatOpenAI` 会直接删掉你设置的其他温度值。所以对推理模型调温度，往往是白费功夫。

**解决办法**：通过 `extra_body` 传入 DeepSeek 特有的参数，关闭思考模式（`extra_body` 的用法见第 7 节）：

```python
extra_body={"thinking": {"type": "disabled"}}
```

## 1.6 关闭思考模式，重新实验

加上 `extra_body` 关闭思考模式，再对比 temperature=0 和 temperature=1.5。这次换一个输出更短、更方便对比的任务：给橘猫起 5 个名字（为什么不继续用写诗，下面会解释）：

In [3]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

for t in [0, 1.5]:
    model = init_chat_model(
        model="deepseek-v4-flash",
        model_provider="deepseek",
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_BASE_URL"),
        temperature=t,
        extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，temperature 才会生效
    )
    print(f"========== temperature={t} ==========")
    for i in range(3):
        print(model.invoke("给一只橘猫起5个名字，用顿号分隔，只输出名字").content)

========== temperature=0 ==========


大橘、橘子、橘座、小橘、阿橘


大橘、橘子、橘座、小橘、阿橘


大橘、橘子、橘座、小橘、阿橘
========== temperature=1.5 ==========


大橘、橘子、橘座、小橘、阿黄


大橘、元宝、橘子、虎斑、金灿


大橘、橘子、橙橙、金宝、阿橘


关闭思考模式后，温度的效果立刻显现：

- `temperature=0`：三次输出**完全相同** → 适合需要稳定、可复现结果的场景；
- `temperature=1.5`：三次输出**各不相同** → 适合需要创意、多样性的场景。

> 🤔 **为什么不继续用写诗做实验？** 写诗时，很多位置都有几个几乎一样好的候选字。比如让 `deepseek-v4-pro` 写诗，实测第一句第一个字的候选中，"东"约 40%、"夭"约 36%。遇到这种"平局"，服务器端计算的微小数值差异就可能让首选翻转，一个字不同，后面就全不同。实测用 `deepseek-v4-pro` 关闭思考、以 temperature=0 写诗，多次调用的结果仍会不同。这就是 1.8 节误区 1 说的"temperature=0 ≠ 100% 确定"。

## 1.7 眼见为实：用 logprobs 查看模型真实的候选词概率

1.3 用的是模拟数据，其实可以让 DeepSeek 直接返回它每一步的候选词和概率：

- `logprobs=True`：返回每个输出 token 的概率（以对数 $\log p$ 的形式返回，用 `math.exp()` 还原成概率）；
- `top_logprobs=5`：同时返回每个位置概率最高的 5 个候选（DeepSeek 最多 20 个）。

下面让模型给橘猫起**一个**名字，只生成**第一个 token**（`max_tokens=1`），对比不同温度下的候选概率：

In [4]:
import math
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

for t in [0, 0.5, 1.0, 1.5, 2.0]:
    model = init_chat_model(
        model="deepseek-v4-flash",
        model_provider="deepseek",
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_BASE_URL"),
        temperature=t,
        max_tokens=1,    # 只生成 1 个 token
        logprobs=True,   # 返回输出 token 的概率（对数形式）
        top_logprobs=5,  # 同时返回概率最高的 5 个候选
        extra_body={"thinking": {"type": "disabled"}},
    )
    response = model.invoke("给一只橘猫起个名字，只输出名字")
    first_token = response.response_metadata["logprobs"]["content"][0]
    candidates = "  ".join(
        f"{c['token']}:{math.exp(c['logprob']):5.1%}" for c in first_token["top_logprobs"]
    )
    print(f"temperature={t:<4} 选中「{first_token['token']}」  候选 → {candidates}")

temperature=0    选中「橘子」  候选 → 橘子:100.0%  小: 0.0%  橘: 0.0%  橙: 0.0%  南瓜: 0.0%


temperature=0.5  选中「橘子」  候选 → 橘子:54.2%  小:30.3%  橘:14.1%  橙: 1.3%  南瓜: 0.0%


temperature=1.0  选中「橘子」  候选 → 橘子:40.5%  小:30.3%  橘:20.7%  橙: 6.3%  南瓜: 0.5%


temperature=1.5  选中「橘子」  候选 → 橘子:33.3%  小:27.9%  橘:14.6%  橙: 7.7%  南瓜: 4.4%


temperature=2.0  选中「橙」  候选 → 橘子:21.5%  小:18.6%  橘:15.4%  橙: 8.5%  南瓜: 2.4%


观察结果（每次运行的数值会略有不同）：

- **temperature=0**：第一名 100%，其余都是 0% → 每次都选第一名，所以输出固定；
- **温度越高，第一名的概率越低，其他候选的概率越高** → 分布越来越"平"，和 1.3 的模拟一致；
- 温度越高，选中非第一名的机会越大（本次运行中 temperature=2.0 选中了第 4 名"橙"），甚至可能选中前 5 名以外的词 → 这就是"温度太高容易离谱"的来源。

另外，候选中的"小""橘"只是名字的**第一个 token**，后面可能接成"小橘""橘座"等 —— 模型是按 token 生成的，而不是按"词"。

> 📌 从 temperature=0 时概率变成 100% 可以看出：DeepSeek 返回的 logprobs 是**经过温度调整后**的概率，所以能直接用它观察温度的效果。

## 1.8 温度怎么设？

**DeepSeek 官方推荐值**（仅在非思考模式下有效）：

| 场景 | 推荐温度 |
|---|:---:|
| 代码生成 / 数学解题 | 0.0 |
| 数据清洗 / 数据分析 | 1.0 |
| 通用对话 | 1.3 |
| 翻译 | 1.3 |
| 创意写作 / 写诗 | 1.5 |

通用原则：**需要"准"就调低，需要"新"就调高。**

**⚠️ 常见误区**

1. **temperature=0 ≠ 100% 确定**：它只保证每一步都选概率最高的词，但服务器端的浮点运算、并行批处理等因素仍会带来微小的数值差异。遇到几个候选概率接近的"平局"时，首选就可能翻转，输出越长越容易出现分叉（见 1.6 中写诗的例子）。只能说"几乎确定"。
2. **不同厂商的取值范围和默认值不同**：DeepSeek、OpenAI 是 0～2（默认 1），Anthropic Claude 是 0～1。换模型时不要照搬数值，以厂商文档为准。
3. **推理（思考）模型往往不支持调温度**：DeepSeek 思考模式会静默忽略，OpenAI GPT-5 推理模型只支持 1。
4. **temperature 和 top_p 通常只调一个**，见第 2 节。

# 2. top_p（核采样）

`top_p` 和 temperature 一样作用于"抽签"环节，但思路不同：temperature 把概率分布**变陡/变平**，top_p 则**直接砍掉长尾**。

做法：把候选词按概率从高到低排序，从第一名开始累加概率，**累计达到 top_p 就停止**，只在这些词里抽签（其余的词直接淘汰）。

仍以"春风拂过\_\_"为例（柳 50%、面 25%、江 15%、山 7%、城 3%）：

| top_p | 保留的候选 | 说明 |
|:---:|---|---|
| 1.0 | 柳、面、江、山、城 | 全部保留，不过滤 |
| 0.9 | 柳、面、江 | 50% + 25% + 15% = 90%，"山""城"被淘汰 |
| 0.5 | 柳 | 只剩一个，每次都选"柳" |

- top_p 越小 → 候选越少 → 输出越保守；top_p 越大 → 候选越多 → 输出越多样；
- **temperature 调整"概率大小"，top_p 决定"候选范围"**；
- 官方建议：**temperature 和 top_p 只调其中一个**，另一个保持默认。

> ⚠️ **DeepSeek V4 的特殊情况**：非思考模式下 top_p 固定为 1.0（传了也会被忽略）；思考模式下只在 0.95～1.0 之间生效。所以用 DeepSeek 时基本不用管 top_p，但 OpenAI、通义千问等模型支持它，需要了解。

# 3. max_tokens（最大输出 token 数）

`max_tokens` 限制模型**最多生成多少个 token**，达到上限就会被截断。先了解一下什么是 token：

1. 基本单位 : 大模型通过分词器（Tokenizer）将文本拆分后的最小语义单元是token（相当于自然语言中的词或字）。不同的模型采用不同的 分词算法 （如BPE、WordPiece），因此同一段文本在不同模型中的Token数量可能不同。

2. 收费依据 ：大语言模型通常也是以token的数量作为其计量（或收费）的依据。
    * 1个中文Token≈1-1.8个汉字，1个英文Token≈3-4个字符
    * Token与字符转化的可视化工具：
        *  OpenAI提供：https://platform.openai.com/tokenizer
        * 百度智能云提供：https://console.bce.baidu.com/support/#/tokenizer

In [11]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    temperature=1.5,
    max_tokens=40,
)

response = model.invoke("帮我讲解一下什么是模型输出时的参数 温度")
print(response)
print(response.content)
#print(response.usage)


content='' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文。用户问“帮我讲解一下什么是模型输出时的参数 温度”。需要解释 temperature 参数在语言模型生成时的作用。应该包括：模型输出 logits -> softmax'} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 40, 'total_tokens': 80, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 40, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 40}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '5cd2713e-5e68-41d7-b584-a22a6e1119c9', 'finish_reason': 'length', 'logprobs': None} id='lc_run--01a0ce97-df3a-73f2-a365-eb2e4a3a33c0-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 40, 'output_tokens': 40, 'tota

## 3.1 🤔 为什么 content 是空的？

上面设置了 `max_tokens=40`，但输出的 `content=''`，回答是空的！看返回结果中的这几处：

- `'reasoning_content': '我们需要回答中文。用户问“帮我讲解一下……'` → 模型处于**思考模式**，先"想"了一段；
- `'completion_tokens': 40`、`'reasoning_tokens': 40` → 40 个 token 的额度**全部花在了思考上**，还没轮到写正式回答；
- `'finish_reason': 'length'` → 生成是因为**达到长度上限被截断**的（正常结束时为 `'stop'`）。

**结论：对思考模型来说，max_tokens = 思考 token + 回答 token。** 设得太小，模型可能"想完了还没来得及说"。上一篇笔记 `01-model-init-online.ipynb` 中 `max_tokens=1024` 时 content 也是空的，原因相同：1024 个 token 全部用在了思考上。

## 3.2 关闭思考模式后再试

In [5]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    max_tokens=40,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

response = model.invoke("帮我讲解一下什么是模型输出时的参数 温度")
print("content:", response.content)
print("finish_reason:", response.response_metadata["finish_reason"])
print("usage_metadata:", response.usage_metadata)

content: 在大型语言模型（如我）的API调用或本地部署中，**温度** 是一个非常重要的参数。简单来说，它控制模型输出的**随机性**或**创造性**。


finish_reason: length
usage_metadata: {'input_tokens': 14, 'output_tokens': 40, 'total_tokens': 54, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


关闭思考模式后，40 个 token 全部用于回答，content 终于有内容了。但回答只开了个头就被截断了（`finish_reason='length'`），后面的讲解都没能输出。

## 3.3 max_tokens 使用建议

1. **判断是否被截断**：检查 `response.response_metadata["finish_reason"]`，`'length'` 表示被截断，`'stop'` 表示正常结束。
2. **不传时的默认值**：DeepSeek 非思考模式 8K、思考模式 64K，最大可设为 384K。
3. **思考模型要留足额度**：max_tokens 包含思考过程的 token，设太小会"只思考、不回答"。
4. **max_tokens 只管输出**：输入 + 输出的总长度还受模型上下文窗口（context length）的限制。
5. **max_tokens 是"硬截断"，不会让回答变精炼**：模型并不知道这个上限，写到一半就会被直接切断。想要简短的回答，应该在提示词里提要求（如"用一句话回答"），max_tokens 只作为控制成本的兜底。

# 4. stop（停止序列）

`stop` 用来指定一个或多个"停止词"：模型**一旦生成这些字符串就立即停止**，并且停止词本身**不会**出现在输出中。DeepSeek 最多支持 16 个停止序列。

常见用途：只要第一行/第一段内容；让输出在某个标记处结束；在 Agent 中遇到 `Observation:` 时停下来，等待工具的执行结果（后续章节会见到）。

In [6]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

question = "从1数到10，用逗号分隔，只输出数字"

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)
print("不设置 stop：", model.invoke(question).content)

model_with_stop = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
    stop=["5"],  # 生成到 "5" 就停止
)
response = model_with_stop.invoke(question)
print("stop=['5']：", response.content)
print("finish_reason:", response.response_metadata["finish_reason"])

不设置 stop： 1,2,3,4,5,6,7,8,9,10


stop=['5']： 1,2,3,4,
finish_reason: stop


可以看到 "5" 本身没有出现在输出中，并且 `finish_reason` 是 `'stop'`（遇到停止序列也算正常结束）。

# 5. presence_penalty / frequency_penalty（重复惩罚）

这两个参数都用来**减少重复**。按 OpenAI 的规范，取值范围为 -2.0～2.0，默认 0（不惩罚）：

| 参数 | 惩罚规则 | 效果 |
|---|---|---|
| `presence_penalty`（存在惩罚） | 某个 token **只要出现过**，就降低它再次出现的概率，与出现次数无关 | 鼓励模型谈论**新话题** |
| `frequency_penalty`（频率惩罚） | 某个 token **出现次数越多**，惩罚越重 | 减少**逐字重复、车轱辘话** |

正值表示惩罚重复，负值表示鼓励重复（很少用）。一般设置在 0.1～1.0 之间，过大可能导致用词古怪。

> ⚠️ **DeepSeek V4 已废弃这两个参数**（官方文档：传了也不会生效），但 OpenAI、通义千问等模型仍然支持。
>
> 这再次说明：**LangChain 只负责把参数"发出去"，参数是否生效由模型厂商决定。** 遇到"参数不起作用"时，第一时间去查厂商的 API 文档。

# 6. timeout 与 max_retries（超时与重试）

调用线上模型本质上是发送 HTTP 请求，网络抖动、服务器繁忙（429 限流、5xx 错误）都可能导致请求失败：

- `timeout`：单次请求最多等待多少秒，超时则报错。不传时默认 **600 秒**（其中建立连接最多等 5 秒）；
- `max_retries`：请求失败（超时、限流、服务器错误等）后**自动重试**的最大次数，每次重试前的等待时间会逐渐变长（指数退避）。不传时默认 **2 次**。

下面把 timeout 设成 0.01 秒故意制造超时，对比重试 0 次和 2 次的耗时：

In [7]:
import time
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

for retries in [0, 2]:
    model = init_chat_model(
        model="deepseek-v4-flash",
        model_provider="deepseek",
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_BASE_URL"),
        timeout=0.01,         # 0.01 秒内必须响应（故意设得极小）
        max_retries=retries,  # 失败后最多重试几次
    )
    start = time.time()
    try:
        model.invoke("你好")
    except Exception as e:
        print(f"max_retries={retries}：{type(e).__name__}: {e}  耗时 {time.time() - start:.2f} 秒")

max_retries=0：OpenAITimeoutError: Request timed out.  耗时 0.01 秒


max_retries=2：OpenAITimeoutError: Request timed out.  耗时 1.27 秒


`max_retries=2` 时多花了 1 秒左右，这就是两次自动重试以及重试前等待的时间。

**使用建议**：生产环境建议显式设置这两个参数，例如 `timeout=60, max_retries=3`（思考模型耗时较长，timeout 要适当调大）。

**扩展：rate_limiter（限流器）**

批量调用模型时，请求太快可能触发厂商的限流（HTTP 429）。LangChain 内置了一个简单的限流器：

```python
from langchain_core.rate_limiters import InMemoryRateLimiter

rate_limiter = InMemoryRateLimiter(
    requests_per_second=1,      # 每秒最多发 1 个请求
    check_every_n_seconds=0.1,  # 每 0.1 秒检查一次能否发送
    max_bucket_size=10,         # 最多允许积攒 10 个请求的突发量
)
model = init_chat_model(..., rate_limiter=rate_limiter)
```

# 7. 厂商特有参数：extra_body 与 reasoning_effort

前面的参数都属于 OpenAI 接口规范里的"标准参数"，LangChain 为它们提供了同名参数。但各厂商还有自己独有的功能（比如 DeepSeek 的思考模式开关），这时就需要其他传参方式：

| 方式 | 适用场景 | 示例 |
|---|---|---|
| `extra_body` | **厂商独有的参数**，原样放进请求体 | `extra_body={"thinking": {"type": "disabled"}}` |
| `model_kwargs` | OpenAI 规范里有、但 LangChain 没有单独定义的参数 | `model_kwargs={"response_format": {"type": "json_object"}}`（JSON 输出模式） |
| `reasoning_effort` | LangChain 已直接支持的"思考力度"参数 | `reasoning_effort="low"` |

> ⚠️ 厂商独有的参数一定要放进 `extra_body`。如果放进 `model_kwargs`，底层的 openai SDK 不认识它，会直接报错：`TypeError: Completions.create() got an unexpected keyword argument 'thinking'`。

DeepSeek 控制思考模式的两种方式：

- `extra_body={"thinking": {"type": "disabled"}}`：关闭思考模式（`"enabled"` 为开启，默认开启）；
- `reasoning_effort`：控制思考力度，可选 `"none"`（不思考）、`"low"`、`"high"`（默认）、`"max"`。

下面对比不同思考力度下，模型消耗的思考 token 数：

In [8]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

for effort in ["none", "low", "high"]:
    model = init_chat_model(
        model="deepseek-v4-flash",
        model_provider="deepseek",
        api_key=os.getenv("DEEPSEEK_API_KEY"),
        base_url=os.getenv("DEEPSEEK_BASE_URL"),
        reasoning_effort=effort,
    )
    response = model.invoke("9.11 和 9.9 哪个大？只回答数字")
    reasoning_tokens = response.usage_metadata.get("output_token_details", {}).get("reasoning", 0)
    print(f"reasoning_effort={effort:<5} 思考 token 数：{reasoning_tokens:<5} 回答：{response.content}")

reasoning_effort=none  思考 token 数：0     回答：9.11


reasoning_effort=low   思考 token 数：81    回答：9.9


reasoning_effort=high  思考 token 数：125   回答：9.9


思考力度越高，消耗的思考 token 越多（更慢、更贵），但复杂问题的回答通常更可靠。本次运行中，不思考（`none`）时模型答错了（"9.11 和 9.9 哪个大"是大模型的经典易错题），开启思考后答对了。

所以：简单任务可以关闭思考，省时省钱；复杂任务（数学、代码、推理）建议开启。

# 8. 参数的设置时机：初始化时 vs 调用时

前面的参数都是在**初始化时**设置的，对这个模型对象的每次调用都生效。如果只想**临时**改变某一次调用的参数，不必重新创建模型：

- **调用时直接传入**：`model.invoke(问题, stop=[...])`，只对这一次调用生效；
- **`bind()` 绑定参数**：`model.bind(stop=[...])` 返回一个预设了参数的新对象，原模型不受影响。`bind()` 同样适用于 `temperature`、`max_tokens` 等生成参数，调用时传入的值会覆盖初始化时的值。

In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="deepseek",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL"),
    extra_body={"thinking": {"type": "disabled"}},
)
question = "从1数到10，用逗号分隔，只输出数字"

# 方式一：调用时直接传入，只对这一次调用生效
print("invoke 时传入 stop=['7']：", model.invoke(question, stop=["7"]).content)

# 方式二：bind() 得到一个预设了参数的新对象
model_stop_at_3 = model.bind(stop=["3"])
print("bind(stop=['3'])：", model_stop_at_3.invoke(question).content)

# 原来的 model 不受影响
print("原 model：", model.invoke(question).content)

invoke 时传入 stop=['7']： 1,2,3,4,5,6,


bind(stop=['3'])： 1,2,


原 model： 1,2,3,4,5,6,7,8,9,10


# 总结

1. **模型参数远不止 temperature 和 max_tokens**，可以分为四类：
    * 连接类：`model`、`model_provider`、`api_key`、`base_url`
    * 生成控制类：`temperature`、`top_p`、`max_tokens`、`stop`、`presence_penalty` / `frequency_penalty`
    * 网络可靠性类：`timeout`、`max_retries`、`rate_limiter`
    * 厂商特有参数：通过 `extra_body`、`reasoning_effort` 等传入
2. **temperature 的本质**：在"按概率抽签"之前，把候选词的概率分布变陡（低温，更稳定）或变平（高温，更多样）。
3. **参数默认为 `None` = 不发送**，由服务器使用自己的默认值。
4. **LangChain 只负责传参，参数是否生效由厂商决定**：DeepSeek V4 思考模式下 temperature 无效、top_p 基本无效、penalty 参数已废弃，而且都不会报错。
5. **思考模型的 max_tokens 包含思考过程**，要留足额度；用 `finish_reason` 判断输出是否被截断。

**常见场景参数速查（以 DeepSeek 为例）**

| 场景 | 推荐设置 |
|---|---|
| 复杂推理（数学、写代码、分析） | 保持思考模式（默认开启），不用调温度，max_tokens 留足 |
| 信息抽取、分类等要求结果稳定的任务 | 关闭思考 + `temperature=0` |
| 日常对话、翻译 | 关闭思考（更快更省）+ `temperature=1.3` |
| 写诗、创意写作 | 关闭思考 + `temperature=1.5` |
| 生产环境 | 显式设置 `timeout`、`max_retries`，批量调用时加 `rate_limiter` |